In [12]:
import json
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

def load_jsonl(
    path: str,
    *,
    encoding: str = "utf-8",
    errors: str = "strict",
    skip_bad_lines: bool = True,
    max_bad_lines: int = 50,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """
    JSONL 파일을 line-by-line로 읽어 List[dict]로 반환.
    - skip_bad_lines=True면 JSON 파싱 실패 라인은 건너뜀
    - stats에는 total_lines, ok_lines, bad_lines, bad_examples 포함
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

    records: List[Dict[str, Any]] = []
    total = ok = bad = 0
    bad_examples = []

    with p.open("r", encoding=encoding, errors=errors) as f:
        for line_no, line in enumerate(f, start=1):
            total += 1
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if not isinstance(obj, dict):
                    # dict가 아닌 JSON도 들어올 수 있으면 이 부분 조정
                    obj = {"_value": obj}
                records.append(obj)
                ok += 1
            except Exception as e:
                bad += 1
                if len(bad_examples) < max_bad_lines:
                    bad_examples.append(
                        {"line_no": line_no, "error": repr(e), "line_head": line[:200]}
                    )
                if not skip_bad_lines:
                    raise

    stats = {
        "path": str(p),
        "total_lines": total,
        "ok_lines": ok,
        "bad_lines": bad,
        "bad_examples": bad_examples,
    }
    return records, stats


path_folio_1 = "/data3/KJE/code/SituW/situW/output/distill_memo/memo_distill_FOLIO_train_gpt-5-nano_20260102_083108_ba87cd_distill_correct.jsonl"
path_folio_2 = "/data3/KJE/code/SituW/situW/output/distill_memo/memo_distill_FOLIO_train_gpt-5-nano_20260102_094748_526671_distill_correct.jsonl"

folio_1, folio1_stats = load_jsonl(path_folio_1)
folio_2, folio2_stats = load_jsonl(path_folio_2)

print("FOLIO1:", folio1_stats)
print("FOLIO2:", folio2_stats)


    # import pandas as pd
    # df_folio1 = pd.DataFrame(folio_1)
    # df_folio2 = pd.DataFrame(folio_2)
    # print(df_folio1.shape, df_folio2.shape)



FOLIO1: {'path': '/data3/KJE/code/SituW/situW/output/distill_memo/memo_distill_FOLIO_train_gpt-5-nano_20260102_083108_ba87cd_distill_correct.jsonl', 'total_lines': 132, 'ok_lines': 132, 'bad_lines': 0, 'bad_examples': []}
FOLIO2: {'path': '/data3/KJE/code/SituW/situW/output/distill_memo/memo_distill_FOLIO_train_gpt-5-nano_20260102_094748_526671_distill_correct.jsonl', 'total_lines': 224, 'ok_lines': 224, 'bad_lines': 0, 'bad_examples': []}


In [14]:
len(folio_1) , len(folio_2)

(132, 224)

In [15]:
folio_1[0].keys()

dict_keys(['id', 'prompt', 'completion', 'gold', 'predicted', 'cost'])

In [16]:
print(folio_1[0]['prompt'])
print(folio_1[0]['completion'])
print(folio_1[0]['gold'])
print(folio_1[0]['predicted'])

Task Description: Given a logical statement problem, analyze and structure it step by step.
For each statement, extract: time, space, causality (accumulated), intention, protagonist (accumulated).
----
Problem:
All people who regularly drink coffee are dependent on caffeine.
People regularly drink coffee, or they don't want to be addicted to caffeine, or both.
No one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.
Rina is either a student who is unaware that caffeine is a drug, or she is not a student and is she aware that caffeine is a drug.
Rina  is either a student who is dependent on caffeine, or she is not a student and not dependent on caffeine.
----
Question:
If Rina either doesn't want to be addicted to caffeine and is unaware that caffeine is a drug, or neither doesn't want to be addicted to caffeine nor is unaware that caffeine is a drug, then Rina doesn't want to be addicted to caffeine and regularly drinks coffee.
----
Reading:

Let's read st